# v1 vs v2 Grand Average Comparison

Quantitative validation that v2 grand averages match v1 within expected bounds.
Run once after regenerating grand averages from a new pipeline version.

Computes per condition:
- Pearson correlation between v1 and v2 grand average waveforms (EEG channels)
- Max absolute difference in µV

## Referebce results (established 15.10.2026)

All correlations ≥ 0.985. Max diff = 1.24µV (active flow highflow target).
Waveform morphology preserved across all conditions.

- **Passive load**: max_diff < 0.002µV, corr = 1.000 — negligible, floating point only
- **Passive flow**: max_diff < 0.12µV, corr > 0.999 — negligible
- **Active load**: max_diff < 0.38µV, corr > 0.990 — small, from corrected epoch drop logic
- **Active flow stim**: max_diff < 1.24µV, corr > 0.985 — expected, corrected epoch drop
  logic has more impact on target ERP (least frequent stimulus, UT task)
- **Active resp-locked**: max_diff < 0.51µV, corr = 1.000 — essentially identical

Differences are scientifically expected from two v2 corrections:
1. Corrected 'epoch after response' drop logic (< 2% of epochs differ)
2. Two passive participants (S04-G4, S14-G6) with corrected full-length recordings

The EEG signal itself is bit-for-bit identical between v1 and v2 at step7.
Game channel fix (forward-fill vs spline) affects only misc channels, not EEG.

Last edit: 15.10.2026 @anguyen

In [7]:
import datetime
import pickle
import numpy as np
import pandas as pd
import mne
from pathlib import Path
from sys import platform

mne.set_log_level('WARNING')

if platform == 'win32':
    V1_ROOTS = {
        'active':  Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives'),
        'passive': Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Passive\derivatives'),
    }
    V2_ROOTS = {
        'active':  Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2'),
        'passive': Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Passive\derivatives_v2'),
    }
else:
    V1_ROOTS = {
        'active':  Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives'),
        'passive': Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives'),
    }
    V2_ROOTS = {
        'active':  Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2'),
        'passive': Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2'),
    }


def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def get_folder(root, paper, subfolder):
    """Return grand average folder, checking both v1 and v2 layouts."""
    p = root / 'outputs' / 'grand_averages' / paper / subfolder
    if p.exists():
        return p
    p = root / 'grand_averages' / paper / subfolder
    if p.exists():
        return p
    return None


def get_pattern(kind, cond):
    """Return glob pattern for a given kind and condition."""
    is_taskdiff = cond == 'taskdiff'
    prefix = '*taskdiff' if is_taskdiff else f'*cond-{cond}'
    if kind == 'stim_raw':
        return f'{prefix}_stim_raw.pkl'
    elif kind == 'stim_diff':
        return f'{prefix}_stim_diff.pkl'
    elif kind == 'resp_raw':
        return f'{prefix}_resp_*.pkl'


# what to compare
COMPARISONS = [
    ('active',  'load', ['simple', 'complex', 'taskdiff'], ['stim_raw', 'stim_diff', 'resp_raw']),
    ('active',  'flow', ['lowflow', 'highflow'],           ['stim_raw', 'stim_diff', 'resp_raw']),
    ('passive', 'load', ['simple', 'complex'],             ['stim_raw', 'stim_diff']),
    ('passive', 'flow', ['lowflow', 'highflow'],           ['stim_raw', 'stim_diff']),
]

In [8]:
results = []

for cohort, paper, conditions, kinds in COMPARISONS:
    v1_root = V1_ROOTS[cohort]
    v2_root = V2_ROOTS[cohort]

    for kind in kinds:
        subfolder = 'diff' if 'diff' in kind else 'raw'
        # taskdiff has its own subfolder

        for cond in conditions:
            is_taskdiff = cond == 'taskdiff'
            v1_folder = get_folder(v1_root, paper, 'taskdiff' if is_taskdiff else subfolder)
            v2_folder = get_folder(v2_root, paper, 'taskdiff' if is_taskdiff else subfolder)

            pattern   = get_pattern(kind, cond)
            v1_files  = sorted(v1_folder.glob(pattern)) if v1_folder else []
            v2_files  = sorted(v2_folder.glob(pattern)) if v2_folder else []

            if not v1_files or not v2_files:
                print(f'[SKIP] {cohort} {paper} {cond} {kind} — '
                      f"v1={'found' if v1_files else 'MISSING'} "
                      f"v2={'found' if v2_files else 'MISSING'}")
                continue

            v1_pkl = load_pkl(v1_files[-1])
            v2_pkl = load_pkl(v2_files[-1])

            print(f"\n{'='*60}")
            print(f'{cohort.upper()} | {paper.upper()} | {cond} | {kind}')
            print(f'  v1: {v1_files[-1].name}')
            print(f'  v2: {v2_files[-1].name}')

            for key in sorted(set(v1_pkl.keys()) | set(v2_pkl.keys())):
                if key not in v1_pkl or key not in v2_pkl:
                    print(f"  [{key}] missing in {'v1' if key not in v1_pkl else 'v2'}")
                    continue

                v1_list = v1_pkl[key]
                v2_list = v2_pkl[key]
                print(f'  [{key}] n: v1={len(v1_list)}  v2={len(v2_list)}')

                if not v1_list or not v2_list:
                    continue

                ga_v1 = mne.grand_average(v1_list)
                ga_v2 = mne.grand_average(v2_list)

                common = [ch for ch in ga_v1.ch_names if ch in ga_v2.ch_names]
                if not common:
                    print('    no common channels — skipping')
                    continue

                d1 = ga_v1.get_data(picks=common)
                d2 = ga_v2.get_data(picks=common)

                if d1.shape == d2.shape:
                    max_diff = np.abs(d1 - d2).max() * 1e6
                    corr     = np.corrcoef(d1.ravel(), d2.ravel())[0, 1]
                    print(f'    max_diff={max_diff:.4f}µV  corr={corr:.6f}')
                    results.append({
                        'cohort':      cohort,
                        'paper':       paper,
                        'condition':   cond,
                        'kind':        kind,
                        'key':         key,
                        'n_v1':        len(v1_list),
                        'n_v2':        len(v2_list),
                        'max_diff_uv': max_diff,
                        'corr':        corr,
                    })
                else:
                    print(f'    shape mismatch: v1={d1.shape} v2={d2.shape}')


ACTIVE | LOAD | simple | stim_raw
  v1: 20260727_active_48p_cond-simple_stim_raw.pkl
  v2: 20261015_active_48p_cond-simple_stim_raw.pkl
  [simple_novel] n: v1=48  v2=48
    max_diff=0.2563µV  corr=0.999579
  [simple_standard] n: v1=48  v2=48
    max_diff=0.2116µV  corr=0.997498
  [simple_target] n: v1=48  v2=48
    max_diff=0.2135µV  corr=0.999544

ACTIVE | LOAD | complex | stim_raw
  v1: 20260727_active_48p_cond-complex_stim_raw.pkl
  v2: 20261015_active_48p_cond-complex_stim_raw.pkl
  [complex_novel] n: v1=48  v2=48
    max_diff=0.1191µV  corr=0.999629
  [complex_standard] n: v1=48  v2=48
    max_diff=0.2278µV  corr=0.990801
  [complex_target] n: v1=48  v2=48
    max_diff=0.1882µV  corr=0.999622

ACTIVE | LOAD | taskdiff | stim_raw
  v1: 20260727_active_48p_taskdiff_stim_raw.pkl
  v2: 20261015_active_48p_taskdiff_stim_raw.pkl
  [novel] n: v1=48  v2=48
    max_diff=0.3052µV  corr=0.998823
  [standard] n: v1=48  v2=48
    max_diff=0.3209µV  corr=0.986307
  [target] n: v1=48  v2=48
   

C:\Users\anguyen\AppData\Local\Temp\ipykernel_6280\3185951423.py:47: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  ga_v2 = mne.grand_average(v2_list)


    max_diff=0.0988µV  corr=0.999760
  [complex_novel] n: v1=47  v2=47


C:\Users\anguyen\AppData\Local\Temp\ipykernel_6280\3185951423.py:47: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  ga_v2 = mne.grand_average(v2_list)


    max_diff=0.0698µV  corr=0.999791
  [complex_standard] n: v1=47  v2=47


...

    max_diff=0.0259µV  corr=0.999904

PASSIVE | LOAD | simple | stim_diff
  v1: 20260727_passive_47p_cond-simple_stim_diff.pkl
  v2: 20261015_passive_47p_cond-simple_stim_diff.pkl
  [simple_deviant-standard] n: v1=47  v2=47
    max_diff=0.0012µV  corr=1.000000
  [simple_novel-standard] n: v1=47  v2=47
    max_diff=0.0011µV  corr=1.000000

PASSIVE | LOAD | complex | stim_diff
  v1: 20260727_passive_47p_cond-complex_stim_diff.pkl
  v2: 20261015_passive_47p_cond-complex_stim_diff.pkl
  [complex_deviant-standard] n: v1=47  v2=47


C:\Users\anguyen\AppData\Local\Temp\ipykernel_6280\3185951423.py:47: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  ga_v2 = mne.grand_average(v2_list)


    max_diff=0.0800µV  corr=0.999348
  [complex_novel-standard] n: v1=47  v2=47


C:\Users\anguyen\AppData\Local\Temp\ipykernel_6280\3185951423.py:47: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  ga_v2 = mne.grand_average(v2_list)


    max_diff=0.0875µV  corr=0.999819

PASSIVE | FLOW | lowflow | stim_raw
  v1: 2026_05_22_passive_41p_cond-lowflow_stim_raw.pkl
  v2: 20261015_passive_41p_cond-lowflow_stim_raw.pkl
  [lowflow_deviant] n: v1=41  v2=41
    max_diff=0.0748µV  corr=0.999725
  [lowflow_novel] n: v1=41  v2=41
    max_diff=0.0747µV  corr=0.999882
  [lowflow_standard] n: v1=41  v2=41
    max_diff=0.0336µV  corr=0.999856

PASSIVE | FLOW | highflow | stim_raw
  v1: 2026_05_22_passive_41p_cond-highflow_stim_raw.pkl
  v2: 20261015_passive_41p_cond-highflow_stim_raw.pkl
  [highflow_deviant] n: v1=41  v2=41
    max_diff=0.0017µV  corr=1.000000
  [highflow_novel] n: v1=41  v2=41
    max_diff=0.0008µV  corr=1.000000
  [highflow_standard] n: v1=41  v2=41
    max_diff=0.0005µV  corr=1.000000

PASSIVE | FLOW | lowflow | stim_diff
  v1: 2026_05_22_passive_41p_cond-lowflow_stim_diff.pkl
  v2: 20261015_passive_41p_cond-lowflow_stim_diff.pkl
  [lowflow_deviant-novel] n: v1=41  v2=41
    max_diff=0.1202µV  corr=0.999613
  [l

In [12]:
# ==============================================================================
# Summary and save
# ==============================================================================

df = pd.DataFrame(results)

print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
if not df.empty:
    print(df.to_string(index=False))
    print(f"\nMin correlation:  {df['corr'].min():.6f}")
    print(f"Max diff (uV):    {df['max_diff_uv'].max():.4f}")
    print(f"All corr > 0.99:  {(df['corr'] > 0.99).all()}")
    print(f"All corr > 0.985: {(df['corr'] > 0.985).all()}")

    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

    for study, root in V2_ROOTS.items():
        df_study = df[df['cohort'] == study]
        if df_study.empty:
            continue

        report = (
            f'v1 vs v2 Grand Average Validation — {study.upper()}\n'
            f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n\n'
            f'Known good results (validated 15.10.2026):\n'
            f'  All correlations >= 0.985. Max diff = 1.24uV (active flow highflow target).\n'
            f'  Passive load: max_diff < 0.002uV, corr = 1.000\n'
            f'  Passive flow: max_diff < 0.12uV,  corr > 0.999\n'
            f'  Active load:  max_diff < 0.38uV,  corr > 0.990\n'
            f'  Active flow:  max_diff < 1.24uV,  corr > 0.985\n'
            f'  Active resp:  max_diff < 0.51uV,  corr = 1.000\n\n'
            f'This run ({study.upper()}):\n'
            f'  Min correlation:  {df_study["corr"].min():.6f}\n'
            f'  Max diff (uV):    {df_study["max_diff_uv"].max():.4f}\n'
            f'  All corr > 0.985: {(df_study["corr"] > 0.985).all()}\n\n'
            + df_study.to_string(index=False)
        )

        out_path = root / 'outputs' / f'v1_vs_v2_validation_{timestamp}.txt'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(report, encoding='utf-8')
        print(f'Saved: {out_path}')


SUMMARY
 cohort paper condition      kind                       key  n_v1  n_v2  max_diff_uv     corr
 active  load    simple  stim_raw              simple_novel    48    48     0.256263 0.999579
 active  load    simple  stim_raw           simple_standard    48    48     0.211622 0.997498
 active  load    simple  stim_raw             simple_target    48    48     0.213453 0.999544
 active  load   complex  stim_raw             complex_novel    48    48     0.119052 0.999629
 active  load   complex  stim_raw          complex_standard    48    48     0.227757 0.990801
 active  load   complex  stim_raw            complex_target    48    48     0.188227 0.999622
 active  load  taskdiff  stim_raw                     novel    48    48     0.305164 0.998823
 active  load  taskdiff  stim_raw                  standard    48    48     0.320947 0.986307
 active  load  taskdiff  stim_raw                    target    48    48     0.259877 0.998802
 active  load    simple stim_diff     simple_novel-